# Task 2.
## Implement a data loader for the BabyLM-Strict-Small data
- You will need a tokenizer (train a byte-level BPE Tokenizer from scratch)
- Take care to use all the sequences in the dataset (many inputs are longer than the maximum context length of the model you will train)

### Training a byte-level BPE Tokenizer from scratch 

**Notes:** By training a tokenizer on raw text we are turning strings into integer IDs. Here, it is important that the `vocab_size` of the model we've implemented so far matches that of the tokenizer's. Then, we encode all the BabyLM-Strict-Small data and use end-of-text tokens. Then, we seperate it into chunks of `max_length` tokens. This is smart, because nothing is discarded. Lastly, we wrap the chunks in a "data loader". This produces a format of `(inputs, targets, mask)`. The mask is not what the model has to guess. It is merely a tensor which tells the model which input positions are real tokens and which are not. So the model knows what is padding. Also, the target `y` is used after the model has predicted the next word, so we can calculate the loss.

_The objective is is turn approx. 1.1M sentences into batches of `(inputs, targets, mask)`. Our GPT-2 model will then train on these batches._

#### 1. Inspecting the data

In [1]:
# Inspecting the data

from datasets import load_dataset

ds = load_dataset("BabyLM-community/BabyLM-2026-Strict-Small")
print(ds)

c:\Users\chris\anaconda3\envs\asds2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 1104106
    })
})


It appears the dataset only has a `train` split, that is, 1M+ rows and one `text` column.

In [2]:
print(ds["train"][0:3])

{'text': ["Well it's just that, you know, a pound, or a hundred pounds today, is not the same as a hundred pounds in a year's time, or two, two years' time.", 'Right.', 'So that would be opportunity cost?']}


It looks like the first three sentences are about economics. Also, it appears we have one line per row.

#### 2. Train a byte-level BPE tokenizer

In [ ]:
from tokenizers import ByteLevelBPETokenizer

VOCAB_SIZE = 8000  # as states in the ATNLP-GPT hyperparameters

texts = ds["train"]["text"] # lists of strings, i.e., the text field of every row 

tokenizer = ByteLevelBPETokenizer() # empty byte-level BPE tokenizer
# "byte-level" means any text, because there are 256 bytes in the base alphabet

# training the empty tokenizer on BabyLM sentences
tokenizer.train_from_iterator(
    texts, # learn merges from this
    vocab_size=VOCAB_SIZE,
    min_frequency=2, # pairs are only merged if they are seen at least twice
    special_tokens=[
        "<pad>", # pads short sequences (ID 0)
        "<|endoftext|>"], # marking the end of each sentence (ID 1)
)

# save to current folder as .json and .txt files
tokenizer.save_model(".", "babylm_bpe") # then we do not have to retrain again

['.\\babylm_bpe-vocab.json', '.\\babylm_bpe-merges.txt']

In [ ]:
from tokenizers import ByteLevelBPETokenizer

# loading the tokenizer
tokenizer = ByteLevelBPETokenizer(
    "babylm_bpe-vocab.json",
    "babylm_bpe-merges.txt"
)

**On BPE:** BPE begins with single bytes and groups common pairs into tokens (e.g. 'm', 'a' and 'n' -> [109, 97, 110] and BPE merges them like m + a -> ma, then ma + n -> man). When the vocabulary has reached 8000 tokens, it stops.

In [4]:
# sanity check

print(tokenizer.get_vocab_size()) # must be 8000 as laid out in the assignment
print(tokenizer.token_to_id("<pad>"), tokenizer.token_to_id("<|endoftext|>")) # ID 0 and 1

# encoding test
enc_test = tokenizer.encode("There is a dog in the tree!")

# Note: BPE writes space using the "Ġ" symbol
print(enc_test.tokens) # how does it tokenize the sentences?
print(enc_test.ids) # what are each words' integer IDs?
print(tokenizer.decode(enc_test.ids)) # must return the original sentence

8000
0 1
['There', 'Ġis', 'Ġa', 'Ġdog', 'Ġin', 'Ġthe', 'Ġtree', '!']
[1041, 327, 260, 1212, 301, 263, 2180, 2]
There is a dog in the tree!


In [ ]:
"""
import torch

# we have two special tokens
eos_id = tokenizer.token_to_id("<|endoftext|>")
pad_id = tokenizer.token_to_id("<pad>")  

texts = [] # collecting all sentences
for row in ds["train"]:
    sentence = row["text"]
    if sentence.strip() != "": # skipping empty rows
        texts.append(sentence)

print(f"There are {len(texts)} sentences.") # should return approx. 1.1M sentences"
"""

There are 1104106 sentences.


In [5]:
import torch

eos_id = tokenizer.token_to_id("<|endoftext|>")
pad_id = tokenizer.token_to_id("<pad>")

texts = ds["train"]["text"]

print(f"There are {len(texts)} sentences.")

There are 1104106 sentences.


There were no empty rows.

#### 3. Encode everything, making one long stream

In [ ]:
# encoding all 1.1M sentences at once (may take a while)

encodings = tokenizer.encode_batch(texts) # this gives us one encoding object per sentence
# sentences are still separable at this point! we just have a list of lists

all_ids = [] # we want just one (long) list of token IDs
# later we feed the stream into a "machine" that seperates it into 256-token chunks

for enc in encodings: # 1.1M iterations
    for token_id in enc.ids: # token IF of a given sentence
        all_ids.append(token_id)
    all_ids.append(eos_id) # mark with ID 1 (eos) after the sentence's tokens

# our model and dataset works with PyTorch tensors
ids = torch.tensor(all_ids, dtype=torch.long) # 64-bit integers

print(f"We have {ids.shape[0]} tokens all together.")

We have 17277036 tokens all together.


In [ ]:
# save the encodings
torch.save(ids, "babylm_ids.pt")

# this file is greater than 100MB and is stored locally on CSN's laptop

In [ ]:
import torch

# loading the token stream
ids = torch.load("babylm_ids.pt")

C:\Users\chris\AppData\Local\Temp\ipykernel_22048\283962935.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ids = torch.load("babylm_ids.pt")


In [23]:
print(ids[:10].tolist()) # first 10 token ids

# if we skip our special tokens, do we then see <|endoftext|>?
print(tokenizer.decode(ids[25:50].tolist(), skip_special_tokens=False)) 

print(ids.max().item() < 8000) # should be True; it finds the largest ID in the entire stream
# .item() turns a one-element tensor into a regular Python number
print((ids == eos_id).sum().item()) # should be 1104106, one per sentence

[824, 315, 309, 539, 321, 13, 291, 406, 13, 260]
 pounds in a year's time, or two, two years' time.<|endoftext|>Right.<|endoftext|>So that would be opportunity cost
True
1104106


#### 4. Divide the stream into chunks

In [10]:
import math

IGNORE = -100 # we do not want the model to train on padding

def make_chunks(ids, block_size, pad_id):
    inputs = ids[:-1] # all but the last token
    targets = ids[1:] # all but the first token

    # necessary number of chunks, leftover tokens get their own chunk because we round up
    num_targets = len(targets)
    num_chunks = math.ceil(num_targets / block_size)

    # necessary padding slots to fill up the last chunk
    total_slots = num_chunks * block_size
    n_pad = total_slots - num_targets

    inputs = torch.cat([inputs, torch.full((n_pad,), pad_id)])
    # torch.cat sticks tensors together end to end
    # torch.full creates a tensor of n copies of v
    targets = torch.cat([targets, torch.full((n_pad,), IGNORE)])
    # while inputs are padded with <pad>, targets are padded with IGNORE (the model should not learn from padding)

    # converting one long one-dimensional tensor into tables where rows are chunks and columns block_size
    x_all = inputs.view(num_chunks, block_size)
    y_all = targets.view(num_chunks, block_size)

    return x_all, y_all

In [ ]:
# now we wrap it in a DS

from torch.utils.data import TensorDataset

BLOCK = 256

x_all, y_all = make_chunks(ids, BLOCK, pad_id)

full_ds = TensorDataset(x_all, y_all)
# this pairs row i of x_all with row i of y_all

#### 5. Batch and split

In [ ]:
# the dataloader, in its current form, receives 32 (x, y) pairs
# we make a function that merges the pairs into a single batch

# this function gather 32 chunks into one batch and builds a padding mask
def collate_fn(batch): # one batch is a list of 32 samples
    all_x = [] # stores the 32 inputs
    all_y = [] # stores the 32 targets
    for x, y in batch:
        all_x.append(x)
        all_y.append(y)

    x = torch.stack(all_x) # [32, 256]
    y = torch.stack(all_y) # [32, 256]

    mask = (x != pad_id).unsqueeze(1).unsqueeze(2) # the mask is True for real tokens and False for <pad>
    # .unsqueeze(1) inserts a new dimension of size 1 at position 1
    # .unsqueeze(2) inserts a new dimension of size 1 at position 2
    # we defined pad_id eariler
    # the shape is [B, 1, 1, T]; what is needed for our MultiHeadAttention model

    return x, y, mask

In [14]:
from torch.utils.data import DataLoader, random_split

# we need to iterate over training and validation

SEED = 42 # ATNLP-GPT hyperparameter
BLOCK = 256 # maximum sequence length, that is, the number of tokens per chunk
BATCH_SIZE = 32 # the number of chunks that goes through the model at once
VAL_FRACTION = 0.05 # this is held out for validation

full_ds = TensorDataset(x_all, y_all)

# 5% of the chunks are randomly assigned to be used for validation, while the rest is used for training
n_val = int(VAL_FRACTION * len(full_ds)) # 5% of the full dataset
n_train = len(full_ds) - n_val # the rest
train_ds, val_ds = random_split( # random_split shuffles chunk indices
    full_ds, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED), # controlling the shuffling so we do not get a different split every run
)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
# wrap the dataset and serve it in batches
# collate_fn, our function from earlier that produces (x, y, mask)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
# not necessary to shuffle validation data

#### 6. Sanity checks

In [ ]:
# how many chunks do we have?
# it should be the number of tokens divided by 256
# also, the training dataset should take up 95% and validation the remaining

print(f"Tokens: {ids.shape[0]}")
print(f"Chunks: {len(full_ds)}")
print(f"Training chunks: {len(train_ds)}")
print(f"Validation chunks: {len(val_ds)}")

Tokens: 17277036
Chunks: 67489
Training chunks: 64115
Validation chunks: 3374


In [16]:
# what is the shape of our batches?

x, y, mask = next(iter(train_dl))

print(x.shape, y.shape, mask.shape)

torch.Size([32, 256]) torch.Size([32, 256]) torch.Size([32, 1, 1, 256])


In [17]:
# also, targets should be the inputs, but shifted by 1
# the following should print True for this to be the case

print(torch.equal(x[0, 1:], y[0, :-1]))

True


In [22]:
# does the text look like normal?

print(tokenizer.decode(x[0].tolist(), skip_special_tokens=False)[:300])

 hotel and threatened to hang them. Only after the pleadings of county attorney N. O. Varnum were the four allowed to leave town.<|endoftext|>U.S. Highway 12 once crossed the St. Croix River on a toll bridge between Wisconsin and Minnesota, which provided revenue for the town. With the construction 


In [19]:
# does the number of IDs fit our vocabulary size of 8000?

print(x.max().item() < 8000, y.max().item() < 8000)

True True


In [ ]:
# the "use all the sequences" requirement

total_targets = 0
for i in range(len(full_ds)):
    x_i, y_i = full_ds[i] # MORE DETAILS
    total_targets += (y_i != IGNORE).sum().item() # do not count padded targets

print(total_targets, ids.shape[0]-1)

17277035 17277035


In [21]:
# is only the last chunk padded?

x_last, y_last = full_ds[len(full_ds)-1]
print(f"Padding of the last chunk: {(x_last == pad_id).sum().item()}")
print(f"We expect: {len(full_ds)*BLOCK-(ids.shape[0]-1)}")

Padding of the last chunk: 149
We expect: 149


In [ ]:
"""
# let's also check whether many inputs actually do exceed the context length

longest = 0
too_long = 0
for enc in encodings:
    n = len(enc.ids)
    if n > longest:
        longest = n
    if n > BLOCK:
        too_long += 1

print(f"Longest row: {longest} tokens, rows over {BLOCK}: {too_long}")
"""